# Empresômetro IA — embeddings dos trechos (BGE-M3)

Este notebook transforma os trechos de texto exportados do banco em vetores, usando o modelo multilíngue **BAAI/bge-m3** com GPU.

**Antes de rodar**

1. No seu computador, gere o arquivo de trechos: `docker compose run --rm documentos exportar-trechos`. Ele fica em `data/embeddings/trechos.csv.gz`, dentro da pasta do projeto.
2. No Google Drive, crie uma pasta chamada `empresometro` e envie esse arquivo para ela.
3. Aqui no Colab, ative a GPU: **Ambiente de execução > Alterar tipo de ambiente de execução > GPU T4**.
4. Rode as células na ordem (**Ambiente de execução > Executar tudo**).

**Resultado:** o arquivo `embeddings.npz`, gravado na mesma pasta do Drive. Baixe-o para `data/embeddings/` no projeto e carregue no banco com `docker compose run --rm documentos importar-embeddings`.

Se a sessão cair no meio, rode tudo de novo: os blocos já calculados são reaproveitados.

In [ ]:
# Configuração
PASTA = "/content/drive/MyDrive/empresometro"
ENTRADA = f"{PASTA}/trechos.csv.gz"
SAIDA = f"{PASTA}/embeddings.npz"
PARCIAIS = f"{PASTA}/parciais"
MODELO = "BAAI/bge-m3"
TAMANHO_DO_LOTE = 64      # reduza para 32 se faltar memória na GPU
BLOCO = 20000             # trechos por arquivo parcial
COMPRIMENTO_MAXIMO = 512  # em tokens; os trechos têm cerca de 300

In [ ]:
# Acesso ao Google Drive e conferência do arquivo de entrada
import os
from google.colab import drive

drive.mount("/content/drive")
assert os.path.exists(ENTRADA), f"Arquivo não encontrado: {ENTRADA}. Envie o trechos.csv.gz para a pasta 'empresometro' do Drive."
print(f"Entrada: {ENTRADA} ({os.path.getsize(ENTRADA) / 1e6:.1f} MB)")

In [ ]:
# Conferência da GPU
import torch

assert torch.cuda.is_available(), "GPU não disponível. Ative em: Ambiente de execução > Alterar tipo de ambiente de execução > GPU T4."
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Instalação da biblioteca de embeddings
!pip install -q -U sentence-transformers

In [ ]:
# Leitura dos trechos
import pandas as pd

trechos = pd.read_csv(ENTRADA, sep=";", dtype={"id": "int64", "texto": "string"}, keep_default_na=False)
assert list(trechos.columns) == ["id", "texto"], f"Colunas inesperadas: {list(trechos.columns)}"
assert trechos["id"].is_unique, "Há identificadores repetidos no arquivo."
print(f"{len(trechos):,} trechos; tamanho mediano de {int(trechos['texto'].str.len().median())} caracteres")
trechos.head(3)

In [ ]:
# Carga do modelo
from sentence_transformers import SentenceTransformer

modelo = SentenceTransformer(MODELO, device="cuda")
modelo.half()                              # meia precisão: mais rápido e usa menos memória
modelo.max_seq_length = COMPRIMENTO_MAXIMO
dimensoes = modelo.get_sentence_embedding_dimension()
assert dimensoes == 1024, f"Dimensão inesperada: {dimensoes}"
print("Modelo carregado; dimensão do vetor:", dimensoes)

In [ ]:
# Geração dos vetores, em blocos gravados no Drive (permite retomar)
import json
import shutil
import time

import numpy as np

os.makedirs(PARCIAIS, exist_ok=True)
marca = {"linhas": int(len(trechos)), "primeiro": int(trechos["id"].iloc[0]), "ultimo": int(trechos["id"].iloc[-1]),
         "bloco": BLOCO, "modelo": MODELO}
caminho_marca = f"{PARCIAIS}/marca.json"
if os.path.exists(caminho_marca) and json.load(open(caminho_marca)) != marca:
    print("O arquivo de entrada mudou desde a última execução; descartando os blocos antigos.")
    shutil.rmtree(PARCIAIS)
    os.makedirs(PARCIAIS)
json.dump(marca, open(caminho_marca, "w"))

inicio = time.time()
feitos = 0
total_de_blocos = (len(trechos) + BLOCO - 1) // BLOCO
for numero, comeco in enumerate(range(0, len(trechos), BLOCO)):
    caminho = f"{PARCIAIS}/bloco_{numero:04d}.npy"
    if os.path.exists(caminho):
        continue
    textos = trechos["texto"].iloc[comeco:comeco + BLOCO].tolist()
    vetores = modelo.encode(
        textos, batch_size=TAMANHO_DO_LOTE, normalize_embeddings=True,
        convert_to_numpy=True, show_progress_bar=True,
    )
    np.save(caminho, vetores.astype("float16"))
    feitos += len(textos)
    ritmo = feitos / (time.time() - inicio)
    print(f"Bloco {numero + 1} de {total_de_blocos} gravado; {ritmo:.0f} trechos por segundo")
print(f"Concluído em {(time.time() - inicio) / 60:.1f} minutos")

In [ ]:
# Junção dos blocos e gravação do arquivo final
blocos = [np.load(f"{PARCIAIS}/bloco_{n:04d}.npy") for n in range(total_de_blocos)]
embeddings = np.concatenate(blocos) if blocos else np.zeros((0, 1024), dtype="float16")
ids = trechos["id"].to_numpy(dtype="int64")
assert embeddings.shape == (len(ids), 1024), f"Formato inesperado: {embeddings.shape}"
assert np.isfinite(embeddings).all(), "Há valores inválidos nos vetores."
np.savez(SAIDA, ids=ids, embeddings=embeddings)
print(f"Arquivo gravado: {SAIDA} ({os.path.getsize(SAIDA) / 1e6:.0f} MB), {len(ids):,} vetores")

In [ ]:
# Conferência: os trechos mais parecidos com uma pergunta de exemplo
pergunta = "Quais são os principais riscos cambiais da companhia?"
vetor = modelo.encode([pergunta], normalize_embeddings=True, convert_to_numpy=True)[0].astype("float32")
semelhanca = embeddings.astype("float32") @ vetor
for posicao in np.argsort(-semelhanca)[:5]:
    print(f"{semelhanca[posicao]:.3f} | id {ids[posicao]} | {trechos['texto'].iloc[posicao][:200]!r}")

## Próximo passo

1. No Google Drive, baixe o arquivo `empresometro/embeddings.npz`.
2. Coloque-o na pasta `data/embeddings/` do projeto, no seu computador.
3. Carregue no banco: `docker compose run --rm documentos importar-embeddings`.

A pasta `parciais` do Drive pode ser apagada depois que a carga der certo.